# sparse training, perturbation, and sparse re-training

The purpose of this notebook is to train a two-layer neural network on sparse data (see its initial accuracy), then perturb the weights in two ways (normal additive noise and normal multiplicative noise; see accuracy), and finally re-train the network. Another objective is to see if the slopes of the training accuracy vs. step curve changes. 

In [17]:
import network
import torch
import torch.nn as nn
import torch.nn.functional as F
from types import SimpleNamespace 
import numpy as np
import matplotlib.pyplot as plt

output_folder = "./eric_sparse_train_preturb_retrain_figures"
seed = 1
network.set_random_mode(True, seed)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") 
store_image = True

[Random Mode] Deterministic with seed 1


# model architecture setup

In [18]:
m = 100 # ADJUST
n = 50 # ADJUST
init_parameters = SimpleNamespace()
init_parameters.m = m 
init_parameters.n = n 
init_parameters.seed = seed
init_parameters.W = torch.randn(n,m).to(device)

# model parameter specification

In [19]:
enose_network = network.enose(init_parameters)
num_data=10000
sparse_training_frac=0.8
dense_training_frac=0.2
is_balanced=True
threshold=0.1 # ADJUST
x_hat = threshold
sparsity = None # ADJUST
num_training = int(num_data*sparse_training_frac)
bounds = (0,1) # ADJUST

# create sparse training dataset

In [20]:
sparse_dataset = network.generate_dataset(m, n, device, num_data, sparse_training_frac, is_balanced, threshold, bounds, sparsity)
sparse_train_conc = sparse_dataset.train_conc
sparse_train_labels = sparse_dataset.train_labels

# create dense testing dataset

In [21]:
dense_dataset = network.generate_dataset(m, n, device, num_data, dense_training_frac, is_balanced, threshold, bounds, None)
dense_test_conc = dense_dataset.test_conc
dense_test_labels = dense_dataset.test_labels

# train two-layer neural net

In [22]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(enose_network.parameters(), lr = 0.001)
lr = 0.001
batch_size = 16
steps = 10000
loss_vec = torch.zeros(steps)
test_interval = 50 
print_interval = 10000 
test_batch_size = 50 
test_iteration_count = 100
accuracy_vec = torch.zeros(steps // test_interval)
loss_vec, accuracy_vec = network.train_model(
    enose_network,
    criterion,
    optimizer,
    lr,
    sparse_train_conc,
    sparse_train_labels,
    dense_test_conc,
    dense_test_labels,
    num_data,
    num_training,
    steps,
    batch_size,
    test_interval,
    print_interval,
    test_batch_size,
    test_iteration_count,
    device
)


[Step 0] Mean Accuracy over 100 Iterations: 0.4862 | Train Loss: 0.9947


# 1. plot training accuracy vs. step (sparse train, dense test)

In [23]:
num_step = len(accuracy_vec)
mean_accuracy = torch.mean(accuracy_vec[num_step//2:-1])

plt.figure(figsize=(8, 4))
plt.plot(torch.arange(0, steps, 50), accuracy_vec, label='Test Accuracy')
plt.axhline(y=0.5, color='red', linestyle=":")
plt.xlabel("Training Step")
plt.ylabel(f"Mean Accuracy w/ {test_iteration_count}")
plt.title(f"Odor Detection Accuracy Over Training, with a mean of {mean_accuracy:.3f}")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
if store_image:
    plt.savefig(f"{output_folder}/accuracy_vs_step_preperturb_sparsity={sparsity}_m={m}_n={n}.pdf")
plt.close()

print(f"Pre-Perturbation Average Test Accuracy during Training: {mean_accuracy}")
print(f"Most Recent Model Test Accuracy: {accuracy_vec[len(accuracy_vec)-1]}")

Pre-Perturbation Average Test Accuracy during Training: 0.7526848316192627
Most Recent Model Test Accuracy: 0.7617999911308289


# 2. matrix transform perturbation

In [24]:
# perturbation parameters
is_multiplicative = False
mean = 0
stddev = 1

enose_network.perturb_weights(True, mean, stddev, is_multiplicative)
new_accuracy_vec = network.eval_model_accuracy(enose_network, 
                                               dense_test_conc, 
                                               dense_test_labels, 
                                               num_data, 
                                               num_training, 
                                               test_batch_size, 
                                               test_iteration_count, 
                                               device)
print(f"Post-Perturbation Average Test Accuracy: {torch.mean(new_accuracy_vec)}")

Post-Perturbation Average Test Accuracy: 0.5055999755859375


# 3. plot retraining accuracy vs. step with sparse retrain set

In [25]:
# create new sparse dataset
sparse_retrain_data = network.generate_dataset(m, n, device, num_data, sparse_training_frac, is_balanced, threshold, bounds, sparsity)
sparse_retrain_conc = sparse_dataset.train_conc
sparse_retrain_labels = sparse_dataset.train_labels

# retrain the network
retrain_loss_vec, retrain_accuracy_vec = network.train_model(
    enose_network,
    criterion,
    optimizer,
    lr,
    sparse_retrain_conc,
    sparse_retrain_labels,
    dense_test_conc,
    dense_test_labels,
    num_data,
    num_training,
    steps,
    batch_size,
    test_interval,
    print_interval,
    test_batch_size,
    test_iteration_count,
    device
)

retrain_num_step = len(retrain_accuracy_vec)
retrain_mean_accuracy = torch.mean(retrain_accuracy_vec[retrain_num_step//2:-1])

plt.figure(figsize=(8, 4))
plt.plot(torch.arange(0, steps, 50), retrain_accuracy_vec, label='Test Accuracy')
plt.axhline(y=0.5, color='red', linestyle=":")
plt.xlabel("Retraining Step")
plt.ylabel(f"Mean Accuracy w/ {test_iteration_count}")
plt.title(f"Odor Detection Accuracy Over Retraining, with a mean of {retrain_mean_accuracy:.3f}")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
if store_image:
    plt.savefig(f"{output_folder}/retrain_accuracy_vs_step_postperturb_sparsity={sparsity}_m={m}_n={n}.pdf")
plt.close()

print(f"Post-Perturbation Average Test Accuracy during Retraining: {retrain_mean_accuracy}")
print(f"Most Recent Model Test Accuracy: {retrain_accuracy_vec[len(retrain_accuracy_vec)-1]}")



[Step 0] Mean Accuracy over 100 Iterations: 0.4990 | Train Loss: 4.0304
Post-Perturbation Average Test Accuracy during Retraining: 0.7345555424690247
Most Recent Model Test Accuracy: 0.7485999464988708
